## Entrenamiento de modelo de ML 

La idea es poder generar varios entrenamientos de 2 maneras de supervision (supervizado o no supervisado).  Veremos como se transforman y actua. 

## Entrenamiento no supervizado (IsolationForest)

se realiza un entrenamiento basico. 


In [4]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np

def cargar(bit):
    """Lee el CSV generado en el notebook 05 para un bit dado."""
    df = pd.read_csv(f'../data/processed/cca_o3_bit{bit:02d}_tasa05.csv',
                     parse_dates=['timestamp'])
    return df[df.received_value.notna()].copy()


def features(df, con_hora=True):
    """Construye la matriz de entrada del modelo.

    Solo received_value y hora. NUNCA original_value, bit, daño ni
    atacado: esas son verdad de campo, no informacion disponible en
    produccion.
    """
    X = pd.DataFrame(index=df.index)
    v = df.received_value
    X['valor_log'] = np.sign(v) * np.log1p(np.abs(v))
    if con_hora:
        X['hora_sin'] = np.sin(2*np.pi*df.hora/24)
        X['hora_cos'] = np.cos(2*np.pi*df.hora/24)
    return X

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score

res = []
for bit in range(13):
    df = cargar(bit)
    X = features(df, con_hora=True)
    y = df.atacado.values          # SOLO para evaluar, nunca para entrenar

    modelo = IsolationForest(contamination=0.05, random_state=42)
    pred = (modelo.fit_predict(X) == -1).astype(int)

    res.append({
        'bit': bit,
        'delta_ppb': 2**bit,
        'recall_%': round(recall_score(y, pred)*100, 1),
        'precision_%': round(precision_score(y, pred, zero_division=0)*100, 1),
        'fp': int(((pred==1) & (y==0)).sum()),
    })

pd.DataFrame(res)